# Bacteria chemotaxis on the NCA manifold

A bacteria colony grows from a single seed cell toward a **food target** on a grid with
**obstacles it must avoid** -- and everything about that journey is decided by external
signals, in the manifold framework of `env_growth.py`:

1. **nutrient `n` in [0,1]** -- a global scalar. The environment encoder maps it onto the
   manifold latent and the DNA decoder generates the update rule's weights,
   `kappa(n) = P(D(Z(n)))`. The nutrient controls **progress**: how close to the target the
   colony's front has advanced. Ramping `n` at eval animates the journey.
2. **attractant `A(x,y)` in [0,1]** -- a spatial field, `A = exp(-D_t / tau)` where `D_t` is
   the **geodesic** distance to the target, computed *around* the obstacles. The attractant
   is **sensed**: it is concatenated to the state before perception, so every cell receives
   `A` and its Sobel gradients and the rule *learns chemotaxis*. Because the field is
   geodesic, climbing its gradient automatically detours around obstacles.
3. **obstacles** -- hard physics, via the chem-gate mechanism: `gate = 1 - obstacle` multiplies
   the whole per-cell state transition, so a cell inside an obstacle is EXACTLY frozen, by
   construction. The rule cannot learn to ignore them.

Unlike the plant notebook, the rule is trained on a **fresh random layout every episode**
(random obstacles, random target, random seed cell) and is verified on **held-out layouts it
has never seen** -- the deliverable is a rule that navigates, not one that memorises scenes.

Design doc: `docs/superpowers/specs/2026-08-30-bacteria-chemotaxis-design.md`


## Setup

In [ ]:
import copy
import math
import os
import time
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

## Configuration

`d_max` doubles as the **progress constant K**: at nutrient `n` the colony occupies the
corridor up to the iso-attractant contour `D_t = (1-n) * K`. The stopping condition is a pure
function of `(A, n)` -- locally decidable by a cell that senses the attractant, with no
knowledge of how long the whole journey is.

In [ ]:
@dataclass
class Config:
    ## cell state
    channels: int = 16
    grid: int = 48
    alive_threshold: float = 0.1
    fire_rate: float = 0.5

    ## update rule
    hidden: int = 128
    leak_init: float = 0.1

    ## the manifold
    latent_dim: int = 8
    dna_hidden: int = 64
    env_hidden: int = 64

    ## environment
    tau: float = 10.0       ## attractant decay: A = exp(-D_t / tau)
    slack: float = 6.0      ## corridor width: D_s + D_t <= D_st + slack
    r0: float = 2.0         ## home-blob radius at the seed (always present)
    d_min: int = 16         ## seed-to-target geodesic range at training time...
    d_max: int = 24         ## ...and d_max is also the progress constant K (see above)
    margin: int = 3         ## seed/target keep-out from the border

    ## training
    epochs: int = 3000
    batch: int = 8          ## layouts per epoch (reduce to 6 if you hit CUDA OOM)
    lr: float = 2e-3
    scalar_lr: float = 1e-4
    n_lo: float = 0.05      ## nutrient is sampled continuously from [n_lo, 1]
    state_noise: float = 0.05
    slow_ramp_prob: float = 0.33
    seed: int = 0
    log_every: int = 100
    eval_every: int = 50    ## fixed held-out eval for best-checkpoint tracking

    ## io
    out_dir: str = "outputs"
    ckpt_name: str = "bacteria.pth"

    @property
    def perception(self) -> int:
        return 3 * (self.channels + 1)      ## +1: the sensed attractant channel


CFG = Config()
os.makedirs(CFG.out_dir, exist_ok=True)
print(CFG)

## Rendering helpers

`composite()` draws the colony's RGBA over the scene: checkerboard agar, the attractant
tinted pale yellow, obstacles dark, and a red ring on the food target.

In [ ]:
def _checkerboard(h, w, tile=2):
    yy, xx = np.mgrid[0:h, 0:w]
    return np.where(((yy // tile) + (xx // tile)) % 2 == 0, 0.74, 0.82)


def composite(state_chw, layout=None):
    """(C,H,W) tensor -> (H,W,3) image. RGBA over agar; the layout (if given) supplies
    the attractant tint, dark obstacles and the target ring."""
    rgba = state_chw[0:4].detach().clamp(0, 1).cpu().numpy()
    rgb, a = rgba[0:3].transpose(1, 2, 0), rgba[3][..., None]
    h, w = rgb.shape[:2]
    bg = _checkerboard(h, w)[..., None] * np.ones(3)
    if layout is not None:
        c = layout.attr[..., None]
        tint = np.array([1.00, 0.96, 0.72])
        bg = bg * (1 - 0.5 * c) + tint * 0.5 * c * bg.mean(axis=2, keepdims=True) * 1.25
        bg = np.clip(bg, 0, 1)
        bg[layout.obstacles] = np.array([0.26, 0.24, 0.28])
    img = np.clip(rgb * a + bg * (1 - a), 0, 1)
    if layout is not None:
        yy, xx = np.mgrid[0:h, 0:w]
        rr = (yy - layout.target[0]) ** 2 + (xx - layout.target[1]) ** 2
        img[(rr >= 2) & (rr <= 6)] = np.array([0.85, 0.13, 0.13])
    return img


def upscale(a, s=5):
    return np.kron(a, np.ones((s, s, 1)))


def frame_with_bar(img, level, bar_h=10):
    """Append a nutrient-level bar below an (H,W,3) image."""
    h, w, _ = img.shape
    bar = np.full((bar_h + 4, w, 3), 0.92)
    fill = int(round(level * (w - 4)))
    bar[2:-2, 2:2 + fill] = np.array([0.30, 0.62, 0.32])
    return np.concatenate([img, bar], axis=0)


def save_gif(frames_rgb, path, duration=70):
    imgs = [Image.fromarray((f * 255).astype(np.uint8)) for f in frames_rgb]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=duration, loop=0)
    print(f"saved {path} ({len(imgs)} frames)")


def save_grid_png(images, ncols, path, labels=None, scale=5, pad=4):
    """Tile (H,W,3) arrays row-major into one PNG."""
    tiles = [upscale(im, scale) for im in images]
    th, tw, _ = tiles[0].shape
    nrows = math.ceil(len(tiles) / ncols)
    canvas = np.ones((nrows * (th + pad) + pad, ncols * (tw + pad) + pad, 3))
    for i, t in enumerate(tiles):
        r, c = divmod(i, ncols)
        y, x = pad + r * (th + pad), pad + c * (tw + pad)
        canvas[y:y + th, x:x + tw] = t
    Image.fromarray((canvas * 255).astype(np.uint8)).save(path)
    print(f"saved {path}" + (f"  [{' | '.join(labels)}]" if labels else ""))

## The environment -- layouts, geodesic attractant, procedural colony targets

A **layout** is obstacles + a target cell + a seed cell. Distances are geodesic
(8-connected BFS over free cells), so the attractant already encodes the detour.

The **ideal colony** at nutrient `n` is a plume along near-shortest paths:
corridor cells (`D_s + D_t <= D_st + slack`) whose remaining distance satisfies
`D_t >= (1-n) * K`, plus a home blob at the seed. The front contour is the
iso-attractant line `A*(n) = exp(-(1-n) K / tau)` -- a pure function of the nutrient,
so a cell can decide *locally* whether to keep advancing. Colour shades from a dark
core (seed side) to a bright front (target side).

In [ ]:
@dataclass
class Layout:
    obstacles: np.ndarray   ## (g,g) bool
    target: tuple           ## (y,x)
    seed: tuple             ## (y,x)
    D_t: np.ndarray         ## geodesic distance to the target (inf where unreachable)
    D_s: np.ndarray         ## geodesic distance to the seed
    D_st: float             ## geodesic seed-to-target distance
    attr: np.ndarray        ## (g,g) float32 attractant, exp(-D_t/tau), 0 on obstacles


def bfs_dist(free, sy, sx):
    """Geodesic (8-connected, unit-cost) distance from (sy,sx) over free cells."""
    dist = np.full(free.shape, np.inf, dtype=np.float32)
    if not free[sy, sx]:
        return dist
    dist[sy, sx] = 0.0
    frontier = np.zeros_like(free)
    frontier[sy, sx] = True
    d = 0
    while frontier.any():
        d += 1
        f = np.zeros_like(frontier)
        f[1:, :] |= frontier[:-1, :];   f[:-1, :] |= frontier[1:, :]
        f[:, 1:] |= frontier[:, :-1];   f[:, :-1] |= frontier[:, 1:]
        f[1:, 1:] |= frontier[:-1, :-1];  f[1:, :-1] |= frontier[:-1, 1:]
        f[:-1, 1:] |= frontier[1:, :-1];  f[:-1, :-1] |= frontier[1:, 1:]
        new = f & free & np.isinf(dist)
        dist[new] = d
        frontier = new
    return dist


def sample_obstacles(g, rng):
    """1-3 shapes from {wall with a gap, blob, bar}; the outer 2-cell ring stays free."""
    obs = np.zeros((g, g), bool)
    for _ in range(int(rng.integers(1, 4))):
        kind = int(rng.integers(0, 3))
        if kind == 0:                                        ## wall with a gap
            pos, th = int(rng.integers(12, g - 12)), int(rng.integers(2, 4))
            gc, gw = int(rng.integers(8, g - 8)), int(rng.integers(3, 6))
            if rng.random() < 0.5:
                obs[pos:pos + th, :] = True
                obs[pos:pos + th, max(0, gc - gw):gc + gw] = False
            else:
                obs[:, pos:pos + th] = True
                obs[max(0, gc - gw):gc + gw, pos:pos + th] = False
        elif kind == 1:                                      ## blob
            cy, cx = rng.integers(8, g - 8, size=2)
            r = int(rng.integers(3, 7))
            yy, xx = np.mgrid[0:g, 0:g]
            obs |= (yy - cy) ** 2 + (xx - cx) ** 2 <= r * r
        else:                                                ## bar
            cy, cx = rng.integers(8, g - 8, size=2)
            L, th = int(rng.integers(8, 16)), int(rng.integers(2, 4))
            if rng.random() < 0.5:
                obs[max(0, cy - L // 2):cy + L // 2, max(0, cx - th // 2):cx + th // 2 + 1] = True
            else:
                obs[max(0, cy - th // 2):cy + th // 2 + 1, max(0, cx - L // 2):cx + L // 2] = True
    obs[:2, :] = False; obs[-2:, :] = False; obs[:, :2] = False; obs[:, -2:] = False
    return obs


def _finish_layout(cfg, obstacles, target, seed_cell):
    free = ~obstacles
    D_t = bfs_dist(free, *target)
    assert np.isfinite(D_t[seed_cell]), "target unreachable from seed"
    D_s = bfs_dist(free, *seed_cell)
    attr = np.where(np.isfinite(D_t), np.exp(-D_t / cfg.tau), 0.0).astype(np.float32)
    return Layout(obstacles, target, seed_cell, D_t, D_s, float(D_t[seed_cell]), attr)


def sample_layout(cfg, rng):
    """Random obstacles, a random target, and a seed whose geodesic distance to the
    target lies in [d_min, d_max]. Rejection-samples until valid."""
    g, m = cfg.grid, cfg.margin
    for _ in range(200):
        obs = sample_obstacles(g, rng)
        free = ~obs
        cand = np.argwhere(free[m:g - m, m:g - m]) + m
        ty, tx = cand[rng.integers(len(cand))]
        D_t = bfs_dist(free, int(ty), int(tx))
        ok = np.isfinite(D_t) & (D_t >= cfg.d_min) & (D_t <= cfg.d_max)
        ok[:m, :] = False; ok[g - m:, :] = False; ok[:, :m] = False; ok[:, g - m:] = False
        cells = np.argwhere(ok)
        if len(cells) == 0:
            continue
        sy, sx = cells[rng.integers(len(cells))]
        return _finish_layout(cfg, obs, (int(ty), int(tx)), (int(sy), int(sx)))
    raise RuntimeError("could not sample a valid layout")


CORE = np.array([0.55, 0.62, 0.22])     ## colony colour, seed side
FRONT = np.array([0.88, 0.92, 0.45])    ## colony colour, target side


def colony_target(layout, n, cfg):
    """RGBA (4,g,g): the ideal colony at nutrient n (see the markdown above)."""
    K = float(cfg.d_max)
    Ds, Dt = layout.D_s, layout.D_t
    fin = np.isfinite(Ds) & np.isfinite(Dt)
    corridor = fin & (Ds + Dt <= layout.D_st + cfg.slack)
    body = (corridor & (Dt >= (1.0 - n) * K)) | (fin & (Ds <= cfg.r0))
    body &= ~layout.obstacles
    t = np.zeros((4,) + Ds.shape, dtype=np.float32)
    shade = np.clip(np.where(fin, Dt, K) / K, 0.0, 1.0)     ## 1 near the seed, 0 at the target
    for c in range(3):
        t[c][body] = FRONT[c] + (CORE[c] - FRONT[c]) * shade[body]
    t[3][body] = 1.0
    return torch.from_numpy(t)


def layout_tensors(layouts, device=DEVICE):
    attr = torch.stack([torch.from_numpy(L.attr) for L in layouts]).unsqueeze(1).to(device)
    gate = torch.stack([torch.from_numpy((~L.obstacles).astype(np.float32))
                        for L in layouts]).unsqueeze(1).to(device)
    return attr, gate


def new_seed(layouts, cfg, device=DEVICE):
    """A single living cell at each layout's seed position."""
    s = torch.zeros(len(layouts), cfg.channels, cfg.grid, cfg.grid, device=device)
    for i, L in enumerate(layouts):
        s[i, 3, L.seed[0], L.seed[1]] = 1.0
    return s


def targets_for(layouts, ns, cfg, device=DEVICE):
    return torch.stack([colony_target(L, float(n), cfg)
                        for L, n in zip(layouts, ns)]).to(device)

In [ ]:
## the data itself, before any training: 4 random layouts, ideal colonies at three
## nutrient levels. If these plumes don't look right, nothing downstream will.
_rng = np.random.default_rng(7)
_Ls = [sample_layout(CFG, _rng) for _ in range(4)]
save_grid_png([composite(colony_target(L, n, CFG), L) for L in _Ls for n in (0.15, 0.5, 1.0)],
              3, os.path.join(CFG.out_dir, "bact_data.png"),
              labels=["rows: 4 random layouts | cols: ideal colony at n = 0.15 / 0.5 / 1.0 "
                      "(obstacles dark, attractant tinted, target ringed)"])
print("D_st per layout:", [round(L.D_st, 1) for L in _Ls])

## The model

`BacteriaNCA` is `EnvNCA` from `env_growth.py` with one change: the attractant is
concatenated to the state **before** perception, so the perception vector carries the
attractant's identity + Sobel gradients (`3*(C+1)` values) and the rule can learn to climb
it. The nutrient->weights manifold path and the hard gate are unchanged: obstacles enter
exactly like `chem`, freezing cells by construction.

In [ ]:
class BacteriaNCA(nn.Module):
    """A manifold NCA that SENSES an attractant field and is HARD-GATED by obstacles.

    kappa(n) = P(D(Z(n))): the nutrient level generates the update net's weights (residual
    on a learned base rule, predictor zero-init). forward() takes the attractant (sensed,
    part of perception) and a gate field (physics, multiplies the whole transition).
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.env_encoder = nn.Sequential(
            nn.Linear(1, cfg.env_hidden), nn.ReLU(),
            nn.Linear(cfg.env_hidden, cfg.latent_dim))

        self.n_generated = cfg.hidden * cfg.perception + cfg.hidden + cfg.channels * cfg.hidden
        self.dna_decoder = nn.Sequential(nn.Linear(cfg.latent_dim, cfg.dna_hidden), nn.ReLU())
        self.predictor = nn.Linear(cfg.dna_hidden, self.n_generated)
        nn.init.zeros_(self.predictor.weight)          ## start as a zero residual: every
        nn.init.zeros_(self.predictor.bias)            ## nutrient decodes to the base rule

        self.base_w1 = nn.Parameter(torch.randn(cfg.hidden, cfg.perception) * 0.001)
        self.base_b1 = nn.Parameter(torch.zeros(cfg.hidden))
        self.base_w2 = nn.Parameter(torch.zeros(cfg.channels, cfg.hidden))

        self.log_leak = nn.Parameter(torch.tensor(math.log(cfg.leak_init)))

        sx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        sy = torch.tensor([[1.0, 2.0, 1.0], [0.0, 0.0, 0.0], [-1.0, -2.0, -1.0]])
        c1 = cfg.channels + 1                          ## +1: the attractant is perceived too
        self.register_buffer("kx", sx.view(1, 1, 3, 3).repeat(c1, 1, 1, 1))
        self.register_buffer("ky", sy.view(1, 1, 3, 3).repeat(c1, 1, 1, 1))

    def weights_for(self, n):
        """(B,) nutrient levels -> the update net's (w1, b1, w2), batched over B."""
        cfg = self.cfg
        z = self.env_encoder(n.view(-1, 1))
        theta = self.predictor(self.dna_decoder(z))
        b, i = z.shape[0], 0
        k = cfg.hidden * cfg.perception
        w1 = self.base_w1 + theta[:, i:i + k].view(b, cfg.hidden, cfg.perception)
        i += k
        b1 = self.base_b1 + theta[:, i:i + cfg.hidden]
        i += cfg.hidden
        w2 = self.base_w2 + theta[:, i:].view(b, cfg.channels, cfg.hidden)
        return w1, b1, w2

    def perceive(self, state, attr):
        x = torch.cat([state, attr], dim=1)                ## the attractant IS sensed
        p = F.pad(x, (1, 1, 1, 1), mode="replicate")       ## no-flux, not a torus
        gx = F.conv2d(p, self.kx, groups=self.cfg.channels + 1)
        gy = F.conv2d(p, self.ky, groups=self.cfg.channels + 1)
        return torch.cat([x, gx, gy], dim=1)

    def alive_mask(self, state):
        return F.max_pool2d(state[:, 3:4], 3, stride=1, padding=1) > self.cfg.alive_threshold

    def forward(self, state, weights, attr, gate=None):
        """One step. attr is (B,1,H,W) sensed attractant; gate is (B,1,H,W) in [0,1] or
        None (== 1 everywhere). gate multiplies the ENTIRE transition, so gate == 0
        freezes a cell exactly -- obstacle cells can never change, by construction."""
        w1, b1, w2 = weights
        pre = self.alive_mask(state)

        p = self.perceive(state, attr)
        a = torch.einsum("bop,bpij->boij", w1, p) + b1[:, :, None, None]
        ds = torch.einsum("bcv,bvij->bcij", w2, F.relu(a))

        m = (torch.rand_like(ds[:, :1]) <= self.cfg.fire_rate).to(ds.dtype)
        out = state + self.log_leak.exp() * ds * m
        new = out * (pre & self.alive_mask(out)).to(out.dtype)

        if gate is None:
            return new
        return state + (new - state) * gate


def rollout(model, state, weights, steps, attr, gate):
    for _ in range(steps):
        state = model(state, weights, attr, gate)
    return state


def rollout_moving(model, state, n_from, n_to, steps, attr, gate, frac=0.6):
    """Rollout under a nutrient IN MOTION: n glides from n_from to n_to over the first
    `frac` of the segment, then holds. Weights are regenerated every step, exactly like
    a ramp at eval -- this makes slow ramps in-distribution instead of a hope."""
    ramp_steps = max(1, int(steps * frac))
    for i in range(steps):
        t = min(1.0, (i + 1) / ramp_steps)
        state = model(state, model.weights_for(n_from + (n_to - n_from) * t), attr, gate)
    return state


def steps_for(n, cfg, extra=24):
    """Enough steps for the front (< 1 cell/step under fire-rate 0.5) to cover the
    worst-case travel that nutrient level n can ask for (n * K cells)."""
    return int(2 * float(n) * cfg.d_max) + extra


def journey_steps(L, extra=40):
    """Enough steps to cover a specific layout's full seed-to-target journey."""
    return int(2 * L.D_st) + extra


@torch.no_grad()
def grow(model, layouts, ns, steps, state=None, seed=0):
    """Fixed-nutrient eval rollout, one row per layout."""
    torch.manual_seed(seed)
    dev = next(model.parameters()).device
    attr, gate = layout_tensors(layouts, dev)
    if state is None:
        state = new_seed(layouts, model.cfg, dev)
    n = torch.as_tensor(ns, dtype=torch.float32, device=dev)
    return rollout(model, state, model.weights_for(n), steps, attr, gate)


@torch.no_grad()
def simulate(model, layout, env_fn, steps, state=None, record_every=1, seed=0):
    """Eval rollout on ONE layout under a TIME-VARYING nutrient env_fn(t) -> n.
    Weights are regenerated whenever the nutrient changes. Reseeds the global RNG."""
    torch.manual_seed(seed)
    dev = next(model.parameters()).device
    attr, gate = layout_tensors([layout], dev)
    if state is None:
        state = new_seed([layout], model.cfg, dev)
    frames, trace = [state.clone()], [float(env_fn(0))]
    cur_n, weights = None, None
    for t in range(steps):
        n = float(env_fn(t))
        if n != cur_n:
            weights = model.weights_for(torch.tensor([n], device=dev))
            cur_n = n
        state = model(state, weights, attr, gate)
        if (t + 1) % record_every == 0:
            frames.append(state.clone())
            trace.append(n)
    return state, frames, trace

## Sanity checks

Before any training time is spent: shapes, exact freezing by the gate (and inside
obstacles), well-formed procedural targets, gradient flow to the predictor and (on a
probe with a non-zero predictor) to the env encoder, and -- new here -- proof that the
attractant actually reaches the update rule.

In [ ]:
def sanity_check(model, cfg, rng):
    dev = next(model.parameters()).device
    layouts = [sample_layout(cfg, rng) for _ in range(4)]
    attr, gate = layout_tensors(layouts, dev)
    ns = torch.tensor([0.2, 0.5, 0.8, 1.0], device=dev)

    w1, b1, w2 = model.weights_for(ns)
    assert w1.shape == (4, cfg.hidden, cfg.perception), w1.shape
    assert b1.shape == (4, cfg.hidden), b1.shape
    assert w2.shape == (4, cfg.channels, cfg.hidden), w2.shape

    state = new_seed(layouts, cfg, dev)
    out = model(state, (w1, b1, w2), attr, gate)
    assert out.shape == state.shape
    assert torch.isfinite(out).all(), "non-finite values after a single step"

    ## gate = 0 must freeze the state EXACTLY
    grown = rollout(model, state, (w1, b1, w2), 8, attr, gate)
    frozen = model(grown, (w1, b1, w2), attr, torch.zeros_like(gate))
    assert (frozen - grown).abs().max().item() == 0.0, "gate=0 did not freeze the state"

    ## obstacle cells must stay EXACTLY zero through a rollout
    many = rollout(model, state, (w1, b1, w2), 30, attr, gate)
    obst = torch.stack([torch.from_numpy(L.obstacles) for L in layouts]).to(dev)
    d_obs = many.abs().max(dim=1).values[obst].max().item() if obst.any() else 0.0
    assert d_obs == 0.0, f"state changed inside an obstacle: {d_obs}"

    ## procedural targets: seed always in the body, target in the body at n=1,
    ## never anything on an obstacle
    for L in layouts:
        t1, t0 = colony_target(L, 1.0, cfg), colony_target(L, cfg.n_lo, cfg)
        assert t1[3, L.target[0], L.target[1]] == 1.0, "target not in the n=1 body"
        assert t0[3, L.seed[0], L.seed[1]] == 1.0, "seed not in the home blob"
        assert (t1[3].numpy() * L.obstacles).max() == 0.0, "target body on an obstacle"

    ## gradient must reach the predictor at init (the thing that moves it off zero)
    final = rollout(model, new_seed(layouts, cfg, dev), (w1, b1, w2), 8, attr, gate)
    loss = F.mse_loss(final[:, 0:4], targets_for(layouts, ns.tolist(), cfg, dev))
    g_pred = torch.autograd.grad(loss, model.predictor.weight)[0].norm().item()
    assert math.isfinite(g_pred) and g_pred > 0, f"grad->predictor {g_pred}"

    ## env-encoder path + attractant sensitivity, probed on a copy with a small
    ## non-zero predictor (at init theta = 0 for every n, so both are legitimately inert)
    probe = copy.deepcopy(model)
    nn.init.normal_(probe.predictor.weight, std=1e-3)
    p_final = rollout(probe, new_seed(layouts, cfg, dev), probe.weights_for(ns), 8, attr, gate)
    g_env = torch.autograd.grad(F.mse_loss(p_final[:, 0:4], targets_for(layouts, ns.tolist(), cfg, dev)),
                                probe.env_encoder[0].weight)[0].norm().item()
    assert math.isfinite(g_env) and g_env > 0, f"grad->env_encoder {g_env}"

    with torch.no_grad():
        torch.manual_seed(3)
        r1 = rollout(probe, new_seed(layouts, cfg, dev), probe.weights_for(ns), 8, attr, gate)
        torch.manual_seed(3)
        r2 = rollout(probe, new_seed(layouts, cfg, dev), probe.weights_for(ns), 8,
                     attr.roll(7, dims=-1), gate)
    d_attr = (r1 - r2).abs().max().item()
    assert d_attr > 0, "shifting the attractant changed nothing -- it is not being sensed"

    n_params = sum(p.numel() for p in model.parameters())
    print(f"sanity: {n_params:,} params | grad->predictor {g_pred:.2e} | "
          f"grad->env_encoder (probe) {g_env:.2e} | attr sensitivity {d_attr:.2e} | "
          f"gate freeze exact | all checks passed")


_m = BacteriaNCA(CFG).to(DEVICE)
sanity_check(_m, CFG, np.random.default_rng(99))
del _m

## Checkpoint save / load

In [ ]:
def save_checkpoint(model, cfg, path=None):
    path = path or os.path.join(cfg.out_dir, cfg.ckpt_name)
    torch.save({"state_dict": model.state_dict(), "config": asdict(cfg)}, path)
    print(f"saved {path}")


def load_checkpoint(path=None, device=DEVICE):
    path = path or os.path.join(CFG.out_dir, CFG.ckpt_name)
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = Config(**ck["config"])
    m = BacteriaNCA(cfg).to(device)
    m.load_state_dict(ck["state_dict"])
    m.eval()
    return m, cfg

## Training -- a fresh random world every episode

Each epoch samples a **new batch of layouts** and runs a two-segment episode on them:

- **segment 1** (from seed, nutrient `a ~ U[0.05, 1]`): loss vs the ideal colony at `a`;
- **segment 2** (from the DETACHED segment-1 state, nutrient `b`): both directions --
  advance AND retreat -- with the state perturbed by noise half the time and the nutrient
  GLIDING there half the time (`rollout_moving`), because at eval the nutrient is a ramp.

A third episode type (probability `slow_ramp_prob`) trains directly on genuine mid-ramp
states via truncated BPTT, exactly as in `env_growth.py`. Rollout lengths scale with the
distance the front actually has to travel. Best-checkpoint tracking uses a FIXED set of
held-out layouts the training rng never sees.

In [ ]:
def _slow_ramp_episode(model, layouts, attr, gate, cfg):
    """Truncated BPTT on a slow nutrient ramp -- the eval distribution itself.
    Rolls the true ramp from seed under no_grad to a random depth (zero memory cost),
    trains a short continuation, then a hold, with loss vs the target at the held level."""
    dev = attr.device
    B = len(layouts)
    lo, hi = cfg.n_lo, 1.0
    tv = int(torch.randint(200, 401, (1,)).item())      ## virtual ramp duration
    ramp_len = max(1, int(0.85 * tv))
    T = int(torch.randint(0, tv - 60, (1,)).item())     ## truncation depth
    descending = torch.rand(1).item() < 0.5             ## downward ramps train retreat

    def rampn(t):
        f = min(1.0, t / ramp_len)
        return hi - (hi - lo) * f if descending else lo + (hi - lo) * f

    state = new_seed(layouts, cfg, dev)
    with torch.no_grad():
        for t in range(T):
            n = torch.full((B,), rampn(t), device=dev)
            state = model(state, model.weights_for(n), attr, gate)

    for t in range(T, T + int(torch.randint(8, 17, (1,)).item())):
        n = torch.full((B,), rampn(t), device=dev)
        state = model(state, model.weights_for(n), attr, gate)

    n_end = rampn(t + 1)
    h = int(torch.randint(24, 41, (1,)).item())         ## hold at the reached level
    state = rollout(model, state, model.weights_for(torch.full((B,), n_end, device=dev)),
                    h, attr, gate)
    return F.mse_loss(state[:, 0:4], targets_for(layouts, [n_end] * B, cfg, dev))


@torch.no_grad()
def _fixed_eval(model, eval_layouts, cfg):
    """Deterministic eval on held-out layouts for best-checkpoint tracking.
    Saves and restores the global RNG so training randomness is untouched."""
    cpu_state = torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        dev = next(model.parameters()).device
        total = 0.0
        for n in (0.6, 1.0):
            state = grow(model, eval_layouts, [n] * len(eval_layouts), steps_for(n, cfg), seed=2)
            total += F.mse_loss(state[:, 0:4],
                                targets_for(eval_layouts, [n] * len(eval_layouts), cfg, dev)).item()
        ramp = lambda t: min(1.0, cfg.n_lo + (1 - cfg.n_lo) * t / 150)
        end, _, _ = simulate(model, eval_layouts[0], ramp, 200, seed=5)
        total += F.mse_loss(end[:, 0:4], targets_for(eval_layouts[:1], [1.0], cfg, dev)).item()
    finally:
        torch.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)
    return total


def train(model, cfg, rng):
    dev = next(model.parameters()).device

    vec = [p for p in model.parameters() if p.numel() > 1]
    sca = [p for p in model.parameters() if p.numel() == 1]
    opt = torch.optim.Adam([{"params": vec, "lr": cfg.lr},
                            {"params": sca, "lr": cfg.scalar_lr}])

    ## held-out layouts for best-checkpoint tracking (a different rng stream than training)
    eval_layouts = [sample_layout(cfg, np.random.default_rng(1234 + i)) for i in range(4)]

    best, best_state = float("inf"), copy.deepcopy(model.state_dict())
    history, t0 = [], time.time()

    for epoch in range(cfg.epochs):
        frac = epoch / max(cfg.epochs - 1, 1)
        opt.param_groups[0]["lr"] = cfg.lr * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))

        layouts = [sample_layout(cfg, rng) for _ in range(cfg.batch)]
        attr, gate = layout_tensors(layouts, dev)
        opt.zero_grad()

        if torch.rand(1).item() < cfg.slow_ramp_prob:
            loss_r = _slow_ramp_episode(model, layouts, attr, gate, cfg)
            loss_r.backward()
            l1, l2 = float("nan"), loss_r.item()
        else:
            B = cfg.batch
            n_a = torch.rand(B, device=dev) * (1 - cfg.n_lo) + cfg.n_lo
            n_b = torch.rand(B, device=dev) * (1 - cfg.n_lo) + cfg.n_lo

            ## segment 1: seed -> the ideal colony at nutrient a
            s1 = steps_for(n_a.max().item(), cfg, extra=int(torch.randint(16, 33, (1,)).item()))
            state = new_seed(layouts, cfg, dev)
            state = rollout(model, state, model.weights_for(n_a), s1, attr, gate)
            loss1 = F.mse_loss(state[:, 0:4], targets_for(layouts, n_a, cfg, dev))
            loss1.backward()

            ## segment 2: nutrient changes to b -- advance or retreat, glide or jump
            s2 = steps_for((n_b - n_a).abs().max().item(), cfg,
                           extra=int(torch.randint(16, 33, (1,)).item()))
            state = state.detach()
            if torch.rand(1).item() < 0.5:             ## half the time: dirty start state
                alive = model.alive_mask(state).to(state.dtype)
                state = state + torch.randn_like(state) * cfg.state_noise * alive
            if torch.rand(1).item() < 0.5:
                state = rollout_moving(model, state, n_a, n_b, s2, attr, gate)
            else:
                state = rollout(model, state, model.weights_for(n_b), s2, attr, gate)
            loss2 = F.mse_loss(state[:, 0:4], targets_for(layouts, n_b, cfg, dev))
            loss2.backward()
            l1, l2 = loss1.item(), loss2.item()

        ## per-parameter gradient normalisation, scalars excluded
        for p in vec:
            if p.grad is not None:
                p.grad /= p.grad.norm() + 1e-8
        opt.step()
        history.append((epoch, l1, l2))

        if epoch % cfg.eval_every == 0 or epoch == cfg.epochs - 1:
            ev = _fixed_eval(model, eval_layouts, cfg)
            if ev < best:
                best, best_state = ev, copy.deepcopy(model.state_dict())

        if epoch % cfg.log_every == 0 or epoch == cfg.epochs - 1:
            print(f"epoch {epoch:5d} | grow {l1:.5f} | switch {l2:.5f} "
                  f"| best-eval {best:.5f} | {time.time() - t0:.0f}s", flush=True)

    model.load_state_dict(best_state)
    print(f"done in {time.time() - t0:.1f}s | best {best:.6f} | weights restored to best")
    return history

In [ ]:
TRAIN = True    ## set False to load outputs/bacteria.pth instead of training
                ## (~40-60 min on an RTX 4060; reduce CFG.batch to 6 on CUDA OOM)

torch.manual_seed(CFG.seed)
if TRAIN:
    model = BacteriaNCA(CFG).to(DEVICE)
    history = train(model, CFG, np.random.default_rng(CFG.seed))
    save_checkpoint(model, CFG)
else:
    model, CFG = load_checkpoint()
model.eval()
print("ready")

## Verification -- the acceptance tests

All on layouts the training rng never produced: 4 fresh random layouts plus 3 hand-built
demos (wall-with-slit, two pillars, and an S-corridor whose journey is LONGER than anything
trained -- a length-generalisation stretch, used for the reached test but not the MSE ones).

In [ ]:
def demo_layouts(cfg):
    g = cfg.grid
    demos = {}

    obs = np.zeros((g, g), bool)                    ## wall with a slit
    obs[:, 23:26] = True
    obs[21:28, 23:26] = False
    demos["slit"] = _finish_layout(cfg, obs, (24, 36), (24, 13))

    obs = np.zeros((g, g), bool)                    ## two pillars with a gap between
    yy, xx = np.mgrid[0:g, 0:g]
    obs = ((yy - 16) ** 2 + (xx - 24) ** 2 <= 30) | ((yy - 32) ** 2 + (xx - 24) ** 2 <= 30)
    demos["pillars"] = _finish_layout(cfg, obs, (24, 36), (24, 12))

    obs = np.zeros((g, g), bool)                    ## S-corridor (longer than trained)
    obs[0:27, 18:21] = True
    obs[21:48, 28:31] = True
    demos["s_corridor"] = _finish_layout(cfg, obs, (23, 39), (23, 9))
    return demos


def decoy_layout(L, cfg):
    """Same obstacles and seed, but the attractant of a DIFFERENT, distant target."""
    free = ~L.obstacles
    D_true = bfs_dist(free, *L.target)
    ok = (np.isfinite(L.D_s) & np.isfinite(D_true)
          & (L.D_s >= cfg.d_min) & (L.D_s <= cfg.d_max) & (D_true >= 14))
    cells = np.argwhere(ok)
    assert len(cells) > 0, "no decoy target available"
    ty, tx = cells[np.random.default_rng(5).integers(len(cells))]
    return _finish_layout(cfg, L.obstacles, (int(ty), int(tx)), L.seed)


def target_alpha(state, L, cfg):
    """Mean alpha in a 2-cell disc around the target -- the 'did it arrive' measure."""
    g = cfg.grid
    yy, xx = np.mgrid[0:g, 0:g]
    disc = ((yy - L.target[0]) ** 2 + (xx - L.target[1]) ** 2 <= 4) & ~L.obstacles
    return float(state[0, 3].clamp(0, 1).detach().cpu().numpy()[disc].mean())


def dilate(mask, r=2):
    """Chebyshev dilation of a bool mask by r cells."""
    out = mask.copy()
    for _ in range(r):
        d = out.copy()
        d[1:, :] |= out[:-1, :];   d[:-1, :] |= out[1:, :]
        d[:, 1:] |= out[:, :-1];   d[:, :-1] |= out[:, 1:]
        d[1:, 1:] |= out[:-1, :-1];  d[1:, :-1] |= out[:-1, 1:]
        d[:-1, 1:] |= out[1:, :-1];  d[:-1, :-1] |= out[1:, 1:]
        out = d
    return out


def verify(model, cfg=CFG):
    dev = next(model.parameters()).device
    results = {}
    rng = np.random.default_rng(777)                ## unseen by training (rng stream 0)
    rand_layouts = [sample_layout(cfg, rng) for _ in range(4)]
    demos = demo_layouts(cfg)

    ## 1. frozen physics: obstacle cells exactly zero after a full journey, and a
    ##    zero gate freezes the whole grown colony exactly
    L = rand_layouts[0]
    state = grow(model, [L], [1.0], journey_steps(L), seed=1)
    obst = torch.from_numpy(L.obstacles).to(dev)
    d_obs = state[0].abs().max(dim=0).values[obst].max().item()
    attr, gate = layout_tensors([L], dev)
    frozen = state
    w = model.weights_for(torch.tensor([1.0], device=dev))
    for _ in range(50):
        frozen = model(frozen, w, attr, torch.zeros_like(gate))
    d0 = (frozen - state).abs().max().item()
    results["freeze"] = (d_obs == 0.0 and d0 == 0.0,
                         f"obstacle |state| {d_obs}, 50 zero-gate steps max|delta| {d0}")

    ## 2. progress stages on unseen random layouts
    levels, worst, det = [0.25, 0.5, 0.75, 1.0], 0.0, []
    for n in levels:
        f = grow(model, rand_layouts, [n] * 4, steps_for(n, cfg, extra=40), seed=2)
        mse = F.mse_loss(f[:, 0:4], targets_for(rand_layouts, [n] * 4, cfg, dev)).item()
        det.append(f"n={n} {mse:.5f}")
        worst = max(worst, mse)
    results["stages"] = (worst < 0.02, f"[{', '.join(det)}], worst {worst:.5f} (< 0.02)")

    ## 3. attractant ablation: a decoy target's field must send the colony elsewhere
    dec = decoy_layout(L, cfg)
    tgt = targets_for([L], [1.0], cfg, dev)
    ok_s = grow(model, [L], [1.0], journey_steps(L), seed=3)
    bad_s = grow(model, [dec], [1.0], journey_steps(L), seed=3)
    mse_ok = F.mse_loss(ok_s[:, 0:4], tgt).item()
    mse_bad = F.mse_loss(bad_s[:, 0:4], tgt).item()
    ratio = mse_bad / max(mse_ok, 1e-12)
    results["ablation"] = (ratio >= 10, f"true field {mse_ok:.6f} -> decoy field "
                                        f"{mse_bad:.6f} ({ratio:.0f}x worse, need >= 10x)")

    ## 4. reached: at n=1 the colony must arrive on EVERY held-out layout, demos included
    alphas = {name: target_alpha(grow(model, [D], [1.0], journey_steps(D), seed=4), D, cfg)
              for name, D in demos.items()}
    for i, R in enumerate(rand_layouts):
        alphas[f"rand{i}"] = target_alpha(grow(model, [R], [1.0], journey_steps(R), seed=4), R, cfg)
    worst_a = min(alphas.values())
    results["reached"] = (worst_a >= 0.5, f"target alpha {({k: round(v, 2) for k, v in alphas.items()})}, "
                                          f"min {worst_a:.2f} (>= 0.5)")

    ## 5. continuous nutrient ramp still arrives (the simulation this exists for)
    Ls = demos["slit"]
    total = journey_steps(Ls, extra=100)
    ramp = lambda t: min(1.0, cfg.n_lo + (1 - cfg.n_lo) * t / int(total * 0.75))
    end, _, _ = simulate(model, Ls, ramp, total, seed=5)
    mse_r = F.mse_loss(end[:, 0:4], targets_for([Ls], [1.0], cfg, dev)).item()
    a_r = target_alpha(end, Ls, cfg)
    results["ramp"] = (mse_r < 0.02 and a_r >= 0.5,
                       f"480-ish-step ramp through the slit: MSE {mse_r:.5f} (< 0.02), "
                       f"target alpha {a_r:.2f} (>= 0.5)")

    ## 6. retreat: after arriving, the nutrient collapses -- the plume must die back
    ##    to the home blob with no ghost left along the abandoned corridor
    Lp = demos["pillars"]
    t1 = journey_steps(Lp)
    def updown(t):
        if t < t1:
            return 1.0
        if t < t1 + 40:
            return round(1.0 - 0.85 * (t - t1) / 40, 3)
        return 0.15
    end, _, _ = simulate(model, Lp, updown, t1 + 40 + int(2 * Lp.D_st) + 60, seed=8)
    tgt = targets_for([Lp], [0.15], cfg, dev)
    mse_re = F.mse_loss(end[:, 0:4], tgt).item()
    ## ghost = residue along the ABANDONED corridor: alpha >= 2 cells away from the ideal
    ## body. The body's own anti-aliased 1-cell edge is already priced into the MSE.
    far = ~dilate(tgt[0, 3].cpu().numpy() > 0, r=2)
    ghost = float(end[0, 3].clamp(0, 1).detach().cpu().numpy()[far].sum())
    results["retreat"] = (mse_re < 0.02 and ghost < 5.0,
                          f"arrive then collapse to n=0.15: MSE {mse_re:.5f} (< 0.02), "
                          f"corridor ghost alpha {ghost:.1f} (< 5)")

    print("\n=== verification (held-out layouts) ===")
    ok = True
    for name, (passed, msg) in results.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {name:<9} {msg}")
        ok &= passed
    print(f"=== {'ALL PASSED' if ok else 'FAILURES PRESENT'} ===\n")
    return ok, results


DEMOS = demo_layouts(CFG)
print("demo journeys (geodesic):", {k: round(v.D_st, 1) for k, v in DEMOS.items()})
ok, results = verify(model, CFG)

## Figures and simulations

Everything below runs on the held-out demo layouts. The nutrient bar under each frame is
the external signal driving progress; the tinted field is the sensed attractant; the dark
shapes are the frozen obstacles; the red ring is the food target.

In [ ]:
## nutrient ramp on each demo: the colony creeps around the obstacles toward the target
for name, L in DEMOS.items():
    total = journey_steps(L, extra=80)
    ramp = lambda t: min(1.0, CFG.n_lo + (1 - CFG.n_lo) * t / int(total * 0.75))
    _, frames, trace = simulate(model, L, ramp, total, record_every=4, seed=5)
    gif = [frame_with_bar(upscale(composite(f[0], L), 4), n) for f, n in zip(frames, trace)]
    save_gif(gif, os.path.join(CFG.out_dir, f"bact_ramp_{name}.gif"))

In [ ]:
## progress stages through the slit: ideal colonies (top) vs grown (bottom)
L = DEMOS["slit"]
levels = [0.25, 0.5, 0.75, 1.0]
imgs = [composite(colony_target(L, n, CFG), L) for n in levels] \
    + [composite(grow(model, [L], [n], steps_for(n, CFG, extra=40), seed=2)[0], L) for n in levels]
save_grid_png(imgs, 4, os.path.join(CFG.out_dir, "bact_stages.png"),
              labels=[f"rows: ideal / grown; cols: n = {levels}"])

In [ ]:
## ablation: the same world driven by the TRUE attractant vs a DECOY target's attractant
L = DEMOS["pillars"]
dec = decoy_layout(L, CFG)
img_ok = composite(grow(model, [L], [1.0], journey_steps(L), seed=3)[0], L)
img_bad = composite(grow(model, [dec], [1.0], journey_steps(L), seed=3)[0], dec)
save_grid_png([img_ok, img_bad], 2, os.path.join(CFG.out_dir, "bact_ablation.png"),
              labels=["left: true attractant; right: decoy attractant (colony provably "
                      "follows the sensed field)"])

In [ ]:
## retreat: arrive at the target, then the nutrient collapses -- the plume dies back
L = DEMOS["pillars"]
t1 = journey_steps(L)
def updown(t):
    if t < t1:
        return 1.0
    if t < t1 + 40:
        return round(1.0 - 0.85 * (t - t1) / 40, 3)
    return 0.15
total = t1 + 40 + int(2 * L.D_st) + 60
_, frames, trace = simulate(model, L, updown, total, record_every=4, seed=8)
gif = [frame_with_bar(upscale(composite(f[0], L), 4), n) for f, n in zip(frames, trace)]
save_gif(gif, os.path.join(CFG.out_dir, "bact_retreat.gif"))

## Recipe: adapt this to your own world

- **Different obstacle families** -- edit `sample_obstacles()`. Whatever you add is
  automatically respected: the BFS attractant routes around it and the gate freezes it.
  Keep the border ring free (or don't -- the demos show obstacles touching the border work).
- **Longer journeys** -- raise `d_min`/`d_max`. `d_max` is also the progress constant `K`,
  so retraining is required; rollout lengths scale automatically via `steps_for()`.
- **A moving target** -- the attractant is just an input: rebuild the layout (or only its
  `attr`) every few steps inside a `simulate()`-style loop and the colony will chase it.
  The rule was trained on a static field, so move the target slowly relative to growth.
- **Multiple food sources** -- take the elementwise max of several targets' attractants.
  Untrained behaviour, but the local stopping rule `A >= A*(n)` often generalises.
- **Interferences that dampen rather than block** -- the gate is continuous: values in
  (0, 1) slow the colony instead of freezing it. Try `gate = 1 - 0.7 * interference`.
